In [11]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report

RANDOM_STATE = 42




train_df = pd.read_csv("train.csv", low_memory=False)
test_df = pd.read_csv("test.csv", low_memory=False)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)




y = train_df["loan_status"].copy()

train_ids = train_df["id"].copy()
test_ids = test_df["id"].copy()

X_train = train_df.drop(columns=["loan_status"], errors="ignore").copy()
X_test = test_df.copy()




cols_to_drop = [
    'out_prncp',
    'out_prncp_inv',
    'total_pymnt',
    'total_pymnt_inv',
    'total_rec_prncp',
    'total_rec_int',
    'total_rec_late_fee',
    'recoveries',
    'collection_recovery_fee',
    'last_pymnt_amnt',
    'last_pymnt_d',
    'next_pymnt_d',
    'last_credit_pull_d',

   
    'grade',
    'sub_grade',
    'int_rate',
    'installment',
    'funded_amnt',
    'funded_amnt_inv'
]



cols_to_drop.append("id")


X_train = X_train.drop(
    columns=cols_to_drop,
    errors="ignore"
)

X_test = X_test.drop(
    columns=cols_to_drop,
    errors="ignore"
)



for df in [X_train, X_test]:

    if "issue_d" in df.columns:

        df["issue_d"] = pd.to_datetime(
            df["issue_d"],
            format="%b-%Y",
            errors="coerce"
        )

        df.drop(columns=["issue_d"], inplace=True)




missing_threshold = 0.5

missing_ratio = X_train.isnull().mean()

cols_high_missing = missing_ratio[
    missing_ratio > missing_threshold
].index.tolist()

print("\nColumns removed because of >50% missing:")

print(cols_high_missing)


X_train = X_train.drop(
    columns=cols_high_missing,
    errors="ignore"
)

X_test = X_test.drop(
    columns=cols_high_missing,
    errors="ignore"
)



numeric_cols = X_train.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

categorical_cols = X_train.select_dtypes(
    include=["object"]
).columns.tolist()


print("\nNumber of numerical columns:",
      len(numeric_cols))

print("Number of categorical columns:",
      len(categorical_cols))





for col in numeric_cols:

    median_value = X_train[col].median()

    X_train[col] = X_train[col].fillna(
        median_value
    )

    X_test[col] = X_test[col].fillna(
        median_value
    )



for col in categorical_cols:

    mode = X_train[col].mode()

    if len(mode) > 0:
        mode_value = mode.iloc[0]
    else:
        mode_value = "Unknown"

    X_train[col] = X_train[col].fillna(
        mode_value
    )

    X_test[col] = X_test[col].fillna(
        mode_value
    )



high_cardinality_cols = [
    col
    for col in categorical_cols
    if X_train[col].nunique() > 50
]

print("\nHigh cardinality columns:")

print(high_cardinality_cols)


X_train = X_train.drop(
    columns=high_cardinality_cols,
    errors="ignore"
)

X_test = X_test.drop(
    columns=high_cardinality_cols,
    errors="ignore"
)


categorical_cols = [
    col
    for col in categorical_cols
    if col not in high_cardinality_cols
]



X_train = pd.get_dummies(
    X_train,
    columns=categorical_cols,
    drop_first=True,
    dtype=int
)

X_test = pd.get_dummies(
    X_test,
    columns=categorical_cols,
    drop_first=True,
    dtype=int
)




X_train, X_test = X_train.align(
    X_test,
    join="left",
    axis=1,
    fill_value=0
)


print("\nAfter encoding:")

print("Train:", X_train.shape)
print("Test :", X_test.shape)




X_train = X_train.astype(float)
X_test = X_test.astype(float)



print("\nTarget distribution:")

print(y.value_counts())

print("\nDefault ratio:")

print(y.mean())



X_tr, X_val, y_tr, y_val = train_test_split(
    X_train,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)




scaler = StandardScaler()

X_tr_scaled = scaler.fit_transform(X_tr)

X_val_scaled = scaler.transform(X_val)




log_reg = LogisticRegression(
    class_weight="balanced",
    random_state=RANDOM_STATE,
    max_iter=1000
)

log_reg.fit(
    X_tr_scaled,
    y_tr
)




y_val_pred = log_reg.predict(
    X_val_scaled
)

y_val_proba = log_reg.predict_proba(
    X_val_scaled
)[:, 1]


print("\nClassification Report:")

print(
    classification_report(
        y_val,
        y_val_pred
    )
)


roc_auc = roc_auc_score(
    y_val,
    y_val_proba
)

print(
    f"\n\n\nValidation ROC-AUC: {roc_auc:.4f}"
)




final_scaler = StandardScaler()

X_train_scaled = final_scaler.fit_transform(
    X_train
)

X_test_scaled = final_scaler.transform(
    X_test
)


final_model = LogisticRegression(
    class_weight="balanced",
    random_state=RANDOM_STATE,
    max_iter=1000
)

final_model.fit(
    X_train_scaled,
    y
)


print("\nFinal model trained successfully!")




test_probability = final_model.predict_proba(
    X_test_scaled
)[:, 1]


print("\nFirst 10 predicted probabilities:")

print(test_probability[:10])




submission = pd.DataFrame({
    "id": test_ids,
    "loan_status": test_probability
})


print("\nSubmission preview:")

print(submission.head())



submission.to_csv(
    "submission.csv",
    index=False
)

print("\nsubmission.csv created successfully!")


Train shape: (52000, 125)
Test shape : (13001, 124)

Columns removed because of >50% missing:
['member_id', 'url', 'desc', 'mths_since_last_delinq', 'mths_since_last_record', 'mths_since_last_major_derog', 'annual_inc_joint', 'dti_joint', 'verification_status_joint', 'open_acc_6m', 'open_act_il', 'open_il_12m', 'open_il_24m', 'mths_since_rcnt_il', 'total_bal_il', 'il_util', 'open_rv_12m', 'open_rv_24m', 'max_bal_bc', 'all_util', 'inq_fi', 'total_cu_tl', 'inq_last_12m', 'mths_since_recent_bc_dlq', 'mths_since_recent_revol_delinq', 'revol_bal_joint', 'sec_app_earliest_cr_line', 'sec_app_inq_last_6mths', 'sec_app_mort_acc', 'sec_app_open_acc', 'sec_app_revol_util', 'sec_app_open_act_il', 'sec_app_num_rev_accts', 'sec_app_chargeoff_within_12_mths', 'sec_app_collections_12_mths_ex_med', 'sec_app_mths_since_last_major_derog', 'hardship_type', 'hardship_reason', 'hardship_status', 'deferral_term', 'hardship_amount', 'hardship_start_date', 'hardship_end_date', 'payment_plan_start_date', 'hards

C:\Users\ASUS\AppData\Local\Temp\ipykernel_6564\1212728270.py:119: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(



After encoding:
Train: (52000, 135)
Test : (13001, 135)

Target distribution:
loan_status
0    41547
1    10453
Name: count, dtype: int64

Default ratio:
0.20101923076923076

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.72      0.79      8309
           1       0.35      0.62      0.45      2091

    accuracy                           0.70     10400
   macro avg       0.62      0.67      0.62     10400
weighted avg       0.78      0.70      0.72     10400




Validation ROC-AUC: 0.7353

Final model trained successfully!

First 10 predicted probabilities:
[0.24512901 0.18278435 0.29567996 0.52292959 0.74104451 0.59486244
 0.45852696 0.46611891 0.34676013 0.24941285]

Submission preview:
      id  loan_status
0  29093     0.245129
1  16268     0.182784
2   3615     0.295680
3   6705     0.522930
4  48573     0.741045

submission.csv created successfully!
